# Minimal Bedrock KB Retrieval Inspector

Use this to fetch and view raw retrieval results (no generation) from your Knowledge Base.


In [1]:
# Setup: imports and env
import os, json, sys
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv
load_dotenv(override=True)

# Ensure repository root (parent of this notebooks folder) is on sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.rag import (
    BedrockKBClient,
    flatten_citations,
    annotate_answer_with_citations,
    extract_text_from_retrieval_result,
    extract_uri,
    extract_title,
 )

print("Repo root:", repo_root)

Repo root: /home/mmark/code/ai-navigator


In [2]:
# Configure AWS + KB
try:
    import yaml
except ImportError:
    yaml = None

CFG_PATH = repo_root / "config" / "config.yaml"
aws_cfg = {}
if yaml and CFG_PATH.exists():
    with open(CFG_PATH, "r", encoding="utf-8") as f:
        cfg_all = yaml.safe_load(f) or {}
        aws_cfg = (cfg_all.get("aws") or {})

REGION = os.getenv("AWS_DEFAULT_REGION") or aws_cfg.get("region")
KB_ID = os.getenv("BEDROCK_KB_ID") or aws_cfg.get("knowledge_base_id")
MODEL_ARN = os.getenv("BEDROCK_MODEL_ARN") or aws_cfg.get("model_arn")
PROFILE_ARN = os.getenv("BEDROCK_INFERENCE_PROFILE_ARN") or aws_cfg.get("inference_profile_arn")

print("Region:", REGION)
print("KB_ID:", KB_ID)
print("Model ARN:", (MODEL_ARN[:20] + "…" if MODEL_ARN else None))
print("Inference Profile ARN:", (PROFILE_ARN[:20] + "…" if PROFILE_ARN else None))

import boto3
brt = boto3.client("bedrock-agent-runtime", region_name=REGION) if REGION else None

Region: us-west-2
KB_ID: IFCZBRIO0U
Model ARN: arn:aws:bedrock:us-w…
Inference Profile ARN: None


In [3]:
# Helpers to inspect retrieval results

def extract_uri(res: dict):
    meta = res.get("metadata") or {}
    for k in (
        "_source_uri",
        "x-amz-kendra-document-id",
        "x-amz-bedrock-kb-source-uri",
        "source",
        "url",
        "URI",
        "uri",
    ):
        if k in meta and meta[k]:
            return str(meta[k])
    loc = res.get("location") or {}
    web = loc.get("webLocation") or {}
    if web.get("url"):
        return str(web["url"])
    s3 = loc.get("s3Location") or {}
    if s3.get("bucket") and s3.get("key"):
        return f"s3://{s3['bucket']}/{s3['key']}"
    return None


def extract_title(res: dict):
    meta = res.get("metadata") or {}
    for k in ("x-amz-kendra-document-title", "title", "documentTitle"):
        if k in meta and meta[k]:
            return str(meta[k])
    return None


def extract_snippet(res: dict):
    content = res.get("content")
    if isinstance(content, list):
        for seg in content:
            if isinstance(seg, dict) and seg.get("text"):
                text = str(seg["text"]).strip()
                return (text[:240] + ("…" if len(text) > 240 else ""))
    if isinstance(content, dict) and content.get("text"):
        text = str(content["text"]).strip()
        return (text[:240] + ("…" if len(text) > 240 else ""))
    return None

In [17]:
# Retrieve and print
QUERY = "what is odi doing with GenAI?"
TOP_K = 5

if not (brt and KB_ID and REGION):
    print("Configure AWS_DEFAULT_REGION and BEDROCK_KB_ID, and ensure credentials are set (e.g., via AWS CLI).")
else:
    resp = brt.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": QUERY},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": TOP_K}},
    )
    results = resp.get("retrievalResults") or []
    print(f"retrievalResults: {len(results)}\n")
    for i, r in enumerate(results, start=1):
        uri = extract_uri(r) or "(no URI)"
        snip = extract_snippet(r) or "(no snippet)"
        print(f"{i}. {snip}\n   {uri}\n")

    # If you want the raw JSON for the first result:
    if results:
        print("First result raw JSON:")
        print(json.dumps(results[0], indent=2, ensure_ascii=False))


retrievalResults: 5

1. Read more ODI and CalHR launch Foundations of GenAI Certificate series 8/1/2024 The California Office of Data and Innovation (ODI) and the California Department of Human Resources (CalHR) are partnering to provide skills and knowledge train…
   https://innovation.ca.gov/blog

2. GenAI.ca.gov is a new digital resource that continues to deliver upon Governor Gavin Newsom’s Executive Order on Generative AI (EO N-12-23). The site answers key questions that state teams may have as they work through the emerging technolo…
   https://innovation.ca.gov/blog/posts/collaboration-and-human-centered-design-key-to-new-genai-website

3. “This is a user-centered site with the dual purpose of illustrating how California continues to lead in implementing GenAI to improve government services and our commitment to the user experience of state staff,” said Office of Data and Inn…
   https://innovation.ca.gov/blog/posts/collaboration-and-human-centered-design-key-to-new-genai-websit

In [19]:
# Parameters
QUERY = "can I sign up for insurance outside of open enrollment"
TOP_K = 5
print("Query:", QUERY, "TopK:", TOP_K)





Query: can I sign up for insurance outside of open enrollment TopK: 5


In [18]:
resp

{'ResponseMetadata': {'RequestId': '014f7c59-425f-4e2e-ae2f-60a45825aaca',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 04 Sep 2025 21:11:48 GMT',
   'content-type': 'application/json',
   'content-length': '9430',
   'connection': 'keep-alive',
   'x-amzn-requestid': '014f7c59-425f-4e2e-ae2f-60a45825aaca'},
  'RetryAttempts': 0},
 'retrievalResults': [{'content': {'text': '\nRead more ODI and CalHR launch Foundations of GenAI Certificate series 8/1/2024 The California Office of Data and Innovation (ODI) and the California Department of Human Resources (CalHR) are partnering to provide skills and knowledge training to prepare state staff to safely deploy generative artificial intelligence (GenAI) tools. A new learning pathway, the Foundations of GenAI Certificate, is available now for current state employees looking to grow … Read more Advancing equity: Bringing heart to data 6/14/2024 Hello, California! I’m thrilled to introduce myself as your new Chief Data Officer (CDO) 

In [5]:
# Run retrieval only
if brt is None:
    print("Configure AWS env and client first in Section 3.")
else:
    ret = brt.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": QUERY},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": int(TOP_K)}}
,    )
    results = ret.get("retrievalResults") or []
    print(f"retrievalResults: {len(results)}")
    for i, r in enumerate(results, start=1):
        uri = extract_uri(r) or "(no URI)"
        title = extract_title(r) or "(no title)"
        snippet = extract_text_from_retrieval_result(r) or "(no snippet)"
        print(f"{i}. {title} — {snippet}\n   {uri}")

retrievalResults: 5
1. Blog | Office of Data and Innovation — Read more ODI and CalHR launch Foundations of GenAI Certificate series 8/1/2024 The California Office of Data and Innovation (ODI) and the California Department of Human Resources (CalHR) are partnering to provide skills and knowledge train…
   https://innovation.ca.gov/blog
2. Collaboration and human-centered design key to new GenAI website | Office of Data and Innovation — GenAI.ca.gov is a new digital resource that continues to deliver upon Governor Gavin Newsom’s Executive Order on Generative AI (EO N-12-23). The site answers key questions that state teams may have as they work through the emerging technolo…
   https://innovation.ca.gov/blog/posts/collaboration-and-human-centered-design-key-to-new-genai-website
3. Collaboration and human-centered design key to new GenAI website | Office of Data and Innovation — “This is a user-centered site with the dual purpose of illustrating how California continues to lead in impleme

In [6]:
# Retrieve_and_generate (compact): answer, spans, first ref fields
if brt is None:
    print("Configure AWS env and client first in Section 3.")
else:
    gen_cfg = {
        "inferenceConfig": {"textInferenceConfig": {"maxTokens": 800, "temperature": 0.2, "topP": 0.9}}
,    }
    prompt = "You are a helpful assistant. Use the provided search results to answer.\n\n$search_results$"
    gen_cfg["promptTemplate"] = {"textPromptTemplate": prompt}
    kb_cfg = {"knowledgeBaseId": KB_ID, "generationConfiguration": gen_cfg}
    if PROFILE_ARN:
        kb_cfg["inferenceProfileArn"] = PROFILE_ARN
    elif MODEL_ARN:
        kb_cfg["modelArn"] = MODEL_ARN
    else:
        raise RuntimeError("Need modelArn or inferenceProfileArn for retrieve_and_generate")
    rag = brt.retrieve_and_generate(
        input={"text": QUERY},
        retrieveAndGenerateConfiguration={"type": "KNOWLEDGE_BASE", "knowledgeBaseConfiguration": kb_cfg}
,    )
    out = rag.get("output", {})
    print("Answer:\n", out.get("text", ""))
    citations = rag.get("citations", []) or []
    print("\nCitations:", len(citations))
    if citations:
        c0 = citations[0]
        gen = (c0.get("generatedResponsePart") or {}).get("textResponsePart") or {}
        print("Span:", gen.get("span"))
        refs = c0.get("retrievedReferences") or []
        if refs:
            r0 = refs[0]
            print("Ref0 URI:", extract_uri(r0))
            print("Ref0 Title:", extract_title(r0))
            print("Ref0 Snippet:", extract_text_from_retrieval_result(r0))

Answer:
 Based on the search results, the California Office of Data and Innovation (ODI) is actively involved in several key GenAI initiatives:

## Leadership and Policy Development
- **Key contributor to GenAI policy**: ODI was instrumental in creating the "Benefits and Risks of the Generative Artificial Intelligence Report" published in November 2023, working alongside other state departments to research beneficial use cases and potential risks of GenAI.

- **Executive Order implementation**: ODI is among more than a dozen state entities tasked with delivering on Governor Newsom's Executive Order on GenAI (EO N-12-23).

## Workforce Training and Development
- **Foundations of GenAI Certificate**: ODI partnered with the California Department of Human Resources (CalHR) to launch a certificate series that provides skills and knowledge training to prepare state staff to safely deploy GenAI tools. ODI's CalAcademy offers one of the courses at no cost to state employees.

- **Human-centere

In [15]:
# print(rag.get("output", []))
# print(rag.get("citations", []))
rag

{'ResponseMetadata': {'RequestId': '5f68cb93-98bd-4459-98ee-7431259b4d42',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Thu, 04 Sep 2025 21:05:10 GMT',
   'content-type': 'application/json',
   'content-length': '2604',
   'connection': 'keep-alive',
   'x-amzn-requestid': '5f68cb93-98bd-4459-98ee-7431259b4d42'},
  'RetryAttempts': 0},
 'citations': [{'generatedResponsePart': {'textResponsePart': {'span': {'end': 50,
      'start': 0},
     'text': 'Sorry, I am unable to assist you with this request.'}},
   'retrievedReferences': []}],
 'output': {'text': 'Based on the search results, the California Office of Data and Innovation (ODI) is actively involved in several key GenAI initiatives:\n\n## Leadership and Policy Development\n- **Key contributor to GenAI policy**: ODI was instrumental in creating the "Benefits and Risks of the Generative Artificial Intelligence Report" published in November 2023, working alongside other state departments to research beneficial use cases and p

In [ ]:
# Retrieve-and-generate: raw response inspection
    resp = brt.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": QUERY},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": TOP_K}},
    )
if not (brt and KB_ID and REGION):
    print("Configure AWS_DEFAULT_REGION and BEDROCK_KB_ID, and ensure credentials are set (e.g., via AWS CLI).")
else:
    # Minimal generation config; adjust as needed
    gen_cfg = {
        "inferenceConfig": {
            "textInferenceConfig": {
                "maxTokens": 800,
                "temperature": 0.2,
                "topP": 0.9,
            }
        }
    }
    # Optional: add a simple prompt that includes $search_results$ to encourage citations
    prompt = "You are a helpful assistant. Use the provided search results to answer.\n\n$search_results$"
    gen_cfg["promptTemplate"] = {"textPromptTemplate": prompt}
    kb_cfg = {
        "knowledgeBaseId": KB_ID,
        "generationConfiguration": gen_cfg,

    }
    if PROFILE_ARN:
        kb_cfg["inferenceProfileArn"] = PROFILE_ARN
    elif MODEL_ARN:
        kb_cfg["modelArn"] = MODEL_ARN
    else:
        raise RuntimeError("Need modelArn or inferenceProfileArn for retrieve_and_generate")
    rag = brt.retrieve_and_generate(
        input={"text": QUERY},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": kb_cfg,
        },
    )
    print("Answer:\n", (rag.get("output", {}) or {}).get("text", ""))
    citations = rag.get("citations", []) or []
    print("\nCitations count:", len(citations))
    for ci, c in enumerate(citations, start=1):
        gen = (c.get("generatedResponsePart") or {}).get("textResponsePart") or {}
        span = gen.get("span") or {}
        print(f"Citation {ci} span:", span)
        refs = c.get("retrievedReferences") or []
        print(f"  refs: {len(refs)}")
        for ri, ref in enumerate(refs[:3], start=1):
            meta = ref.get("metadata") or {}
            content = ref.get("content") or {}
            # Compute best-effort fields from raw ref
            title = meta.get("x-amz-kendra-document-title") or meta.get("title")
            uri = meta.get("_source_uri") or meta.get("x-amz-kendra-document-id") or meta.get("x-amz-bedrock-kb-source-uri")
            if not uri:
                loc = ref.get("location") or {}
                web = loc.get("webLocation") or {}
                if web.get("url"):
                    uri = str(web["url"])
                s3 = loc.get("s3Location") or {}
                if not uri and s3.get("bucket") and s3.get("key"):
                    uri = f"s3://{s3['bucket']}/{s3['key']}"
            snippet = None
            if isinstance(content, dict) and content.get("text"):
                snippet = str(content["text"]).strip()
                snippet = snippet[:240] + ("…" if len(snippet) > 240 else "")
            print(f"    Ref {ri}: title={title!r} uri={uri!r}")
            if snippet:
                print(f"      snippet: {snippet}")

Answer:
 Based on the search results, the California Office of Data and Innovation (ODI) is actively involved in several key GenAI initiatives:

## Leadership and Policy Development
- ODI was a **key contributor** to the "Benefits and Risks of the Generative Artificial Intelligence Report" published in November 2023, working alongside other state departments
- They played a crucial role in **community engagement efforts** and research for this historic report
- ODI helped establish **guardrails and engagement protocols** for GenAI use by state government entities

## Workforce Training and Development
- **Partnering with CalHR** to launch the "Foundations of GenAI Certificate" series to upskill current state staff
- ODI's **CalAcademy program** offers one of the courses in this certificate series at no cost to state employees
- Using **data-driven human-centered design practices** to improve service delivery for Californians

## Digital Resources and User Experience
- **Collaborating w

In [36]:
# Processed output using app/rag.py helpers (annotate + flatten)
from importlib import reload
import sys
from pathlib import Path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
try:
    from app import rag as module_under_test
    module_under_test = reload(module_under_test)
    annotate_answer_with_citations = module_under_test.annotate_answer_with_citations
    flatten_citations = module_under_test.flatten_citations
except Exception as e:
    module_under_test = None
    print("Failed to import app.rag:", e)

if not (brt and KB_ID and REGION):
    print("Configure AWS_DEFAULT_REGION and BEDROCK_KB_ID first.")
elif module_under_test is None:
    pass
else:
    # Re-run retrieve_and_generate to ensure fresh data
    gen_cfg = {
        "inferenceConfig": {
            "textInferenceConfig": {
                "maxTokens": 800,
                "temperature": 0.2,
                "topP": 0.9,
            }
        }
    }
    prompt = "You are a helpful assistant. Use the provided search results to answer.\n\n$search_results$"
    gen_cfg["promptTemplate"] = {"textPromptTemplate": prompt}
    kb_cfg = {
        "knowledgeBaseId": KB_ID,
        "generationConfiguration": gen_cfg,
    }
    rag = brt.retrieve_and_generate(
        input={"text": QUERY},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": kb_cfg,
        },
    )
    answer = (rag.get("output", {}) or {}).get("text", "")
    citations = rag.get("citations", []) or []
    print("Answer (raw):\n", answer)
    enriched, span_refs = annotate_answer_with_citations(answer, citations)
    used_spans = enriched is not None
    if not used_spans:
        # Flatten and show for debugging if spans missing
        refs = flatten_citations(citations)
        print("\nNo spans present. Flattened refs:")
        for r in refs:
            print(f"[{r['index']}] title={r.get('title')} uri={r.get('uri')} snippet={(r.get('snippet') or '')[:120]}")
    else:
        print("\nEnriched answer (span-based):\n", enriched)
        print("\nSpan refs:")
        for r in span_refs:
            print(f"[{r['index']}] title={r.get('title')} uri={r.get('uri')} snippet={(r.get('snippet') or '')[:120]}")

ParamValidationError: Parameter validation failed:
Missing required parameter in retrieveAndGenerateConfiguration.knowledgeBaseConfiguration: "modelArn"

In [ ]:
# pip install boto3
import os
import boto3

REGION = os.getenv("AWS_DEFAULT_REGION", "us-west-2")

# REQUIRED: set these
KB_ID = os.getenv("BEDROCK_KB_ID")
MODEL_ARN = os.getenv("BEDROCK_MODEL_ARN")

brt = boto3.client("bedrock-agent-runtime", region_name=REGION)

def ask_kb(query: str, max_tokens: int = 800, temperature: float = 0.0, top_p: float = 0.9,
           number_of_results: int = 5, override_search_type: str | None = None):
    """
    Sends a grounded RAG request to Bedrock Knowledge Bases and returns:
      - response text
      - a list of citations with snippet + location (URL/URI) + metadata
    """

    req = {
        "input": {"text": query},
        "retrieveAndGenerateConfiguration": {
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": KB_ID,
                "modelArn": MODEL_ARN,
                # Control how the model generates text
                "generationConfiguration": {
                    "inferenceConfig": {
                        "textInferenceConfig": {
                            "maxTokens": max_tokens,
                            "temperature": temperature,
                            "topP": top_p,
                            # optional: "stopSequences": ["</answer>"]
                        }
                    },
                    # Optional: strongly nudge grounding in the prompt
                    "promptTemplate": {
                        "textPromptTemplate": """\
                    You are a helpful assistant. Answer the user's question **only** using the retrieved source chunks.
                    If the sources don't contain the answer, say you don't know.

                    User question:
                    $query$

                    Retrieved source chunks:
                    $search_results$

                    Instructions:
                    - Cite the specific sources you used with bracketed numbers like [1], [2].
                    - Keep the answer concise and factual.

                    $output_format_instructions$
                    """
                    }},
                    # Optional: guardrail
                    # "guardrailConfiguration": {"guardrailId": "<id>", "guardrailVersion": "<ver>"},
                },
                # Control retrieval behavior (chunk count, filters, search type, reranking, etc.)
                "retrievalConfiguration": {
                    "vectorSearchConfiguration": {
                        "numberOfResults": number_of_results,
                        # Optional: restrict to HYBRID or SEMANTIC (defaults to auto if omitted)
                        **({"overrideSearchType": override_search_type} if override_search_type else {})
                        # Optional filter example:
                        # "filter": {"equals": {"key": "category", "value": "policy"}}
                    },
                    # Optional reranker example:
                    # "rerankingConfiguration": {
                    #   "type": "BEDROCK_RERANKING_MODEL",
                    #   "modelConfiguration": {"modelArn": "<reranker-model-arn>"}
                    # }
                }
            }
        }
    }

    resp = brt.retrieve_and_generate(**req)

    answer = (resp.get("output") or {}).get("text", "")
    citations = resp.get("citations", [])  # list of segments with retrievedReferences

    # Flatten citations into a simple list of sources in the order they appear.
    flattened_sources = []
    seen = set()
    for c in citations:
        for ref in c.get("retrievedReferences", []):
            loc = ref.get("location", {})
            # Prefer web/sharepoint/confluence/salesforce URLs, then Kendra/S3/SQL fallbacks
            url = (
                (loc.get("webLocation") or {}).get("url") or
                (loc.get("sharePointLocation") or {}).get("url") or
                (loc.get("confluenceLocation") or {}).get("url") or
                (loc.get("salesforceLocation") or {}).get("url") or
                (loc.get("kendraDocumentLocation") or {}).get("uri") or
                (loc.get("s3Location") or {}).get("uri") or
                (loc.get("sqlLocation") or {}).get("query") or
                ""
            )
            snippet = ((ref.get("content") or {}).get("text") or "").strip()
            key = (url, snippet)
            if key not in seen:
                seen.add(key)
                flattened_sources.append({"url": url, "snippet": snippet, "metadata": ref.get("metadata", {})})

    # Assemble a numbered bibliography to map to [1], [2] in the output
    bib_lines = []
    for i, s in enumerate(flattened_sources, start=1):
        short_snippet = (s["snippet"][:180] + "…") if len(s["snippet"]) > 180 else s["snippet"]
        bib_lines.append(f"[{i}] {s['url'] or '(no URL)'} — {short_snippet}")

    return {
        "answer": answer,
        "sources": flattened_sources,
        "bibliography": "\n".join(bib_lines),
        "sessionId": resp.get("sessionId"),
        "guardrailAction": resp.get("guardrailAction"),
    }

if __name__ == "__main__":
    result = ask_kb("What are the prerequisites and steps to submit a disaster assistance claim?")
    print("\n=== Grounded Answer ===\n")
    print(result["answer"])
    print("\n=== Sources ===")
    print(result["bibliography"] or "(no citations returned)")


ValidationException: An error occurred (ValidationException) when calling the RetrieveAndGenerate operation: [textPromptTemplate::generationConfiguration] must contain the $search_results$ placeholder variable.